# **Task 1: Conceptual Understanding**

**1. Difference between "Love" and "love?**

In raw NLP, these are treated as two distinct tokens because computers use ASCII/Unicode values where 'L' ($76$) differs from 'l' ($108$). If we don't lowercase text, a model might fail to realize they carry the same semantic meaning, leading to a fragmented vocabulary.

**2. What happens if stopwords are not removed?**

If stopwords (e.g., "the", "is", "at") are not removed, they can dominate the frequency distribution. Because they appear so often, they may drown out the "signal" (meaningful words) in tasks like document classification or topic modeling.

**3. Scenarios where removing stopwords is harmful**

* Sentiment Analysis: Removing "not" from "I am not happy" changes the sentiment from negative to positive.

* Machine Translation: Stopwords provide the grammatical structure (syntax) necessary to translate a sentence accurately from one language to another.

**4. Stemming vs. Lemmatization**

* Stemming: A "brute force" approach that chops off the ends of words (e.g., "studies" $\rightarrow$ "studi"). It is fast but often results in non-dictionary words.
* Lemmatization: Uses vocabulary and morphological analysis to return the dictionary base form (lemma) (e.g., "studies" $\rightarrow$ "study"). It is more accurate but computationally slower.

# **Task 2:Build Advanced Preprocessing Function**

In [1]:
def reduce_repeats(word: str) -> str:
    result = []
    for char in word:
        # If the last two characters are the same as current, skip
        if len(result) >= 2 and result[-1] == char and result[-2] == char:
            continue
        result.append(char)
    return "".join(result)


def preprocess_text(text: str) -> str:
    # 1. skip URLs and email-like patterns
    words = text.split()
    cleaned_words = []
    for word in words:
        if word.startswith("http") or word.startswith("www") or "@" in word:
            continue
        cleaned_words.append(word)
    text = " ".join(cleaned_words)

    # skip numbers
    text = "".join([ch for ch in text if not ch.isdigit()])

    # lowercase
    text = text.lower()

    # Handle repeated characters
    words = text.split()
    words = [reduce_repeats(word) for word in words]

    # Skip short tokens (≤ 2), except 'no' and 'not'
    meaningful = {"no", "not"}
    words = [word for word in words if len(word) > 2 or word in meaningful]

    # Remove extra spaces
    text = " ".join(words).strip()
    return text

sample_message = "WOW!!! This is soooo goooood!!! Visit http://example.com now. I have 3 dogs."
print(preprocess_text(sample_message))


wow!! this soo good!! visit now. have dogs.


# **Task 3: Stress Testing**

In [2]:
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

for i, sentence in enumerate(sample_inputs, start=1):
    cleaned_sentence = preprocess_text(sentence)
    cleaned_tokens = cleaned_sentence.split()
    print(f"Test {i}")
    print("Original Text :", sentence)
    print("Cleaned Tokens:", cleaned_tokens)
    print("Cleaned Sentence:", cleaned_sentence)
    print("-" * 50)

Test 1
Original Text : Get 100% FREE access now!!!
Cleaned Tokens: ['get', 'free', 'access', 'now!!']
Cleaned Sentence: get free access now!!
--------------------------------------------------
Test 2
Original Text : I absolutely looooved this product 😍😍
Cleaned Tokens: ['absolutely', 'looved', 'this', 'product']
Cleaned Sentence: absolutely looved this product
--------------------------------------------------
Test 3
Original Text : Worst service ever... 0/10
Cleaned Tokens: ['worst', 'service', 'ever..']
Cleaned Sentence: worst service ever..
--------------------------------------------------
Test 4
Original Text : Call me at 9876543210
Cleaned Tokens: ['call']
Cleaned Sentence: call
--------------------------------------------------
Test 5
Original Text : This is THE best course!!!
Cleaned Tokens: ['this', 'the', 'best', 'course!!']
Cleaned Sentence: this the best course!!
--------------------------------------------------
Test 6
Original Text : Visit https://openai.com now!
Cleaned 

# **Task 4: Token Analytics**

In [3]:
analytics = []

for i, sentence in enumerate(sample_inputs, start=1):
    cleaned_sentence = preprocess_text(sentence)
    tokens = cleaned_sentence.split()
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_token_length = sum(len(t) for t in tokens) / total_tokens if total_tokens > 0 else 0

    analytics.append({
        "index": i,
        "original": sentence,
        "cleaned": cleaned_sentence,
        "tokens": tokens,
        "total": total_tokens,
        "unique": unique_tokens,
        "avg_len": round(avg_token_length, 2)
    })

for item in analytics:
    print(f"Test {item['index']}")
    print("Original Text :", item['original'])
    print("Cleaned Tokens:", item['tokens'])
    print("Cleaned Sentence:", item['cleaned'])
    print("Total Tokens   :", item['total'])
    print("Unique Tokens  :", item['unique'])
    print("Average Length :", item['avg_len'])
    print("-" * 50)

# Which sentence had the most noise? (fewest tokens after cleaning)
most_noise = min(analytics, key=lambda x: len(x['tokens']))
print("Sentence with most noise removed:", most_noise['original'])

# Which sentence retained the most meaningful tokens? (most tokens after cleaning)
most_meaningful = max(analytics, key=lambda x: x['total'])
print("Sentence with most meaningful tokens:", most_meaningful['original'])

Test 1
Original Text : Get 100% FREE access now!!!
Cleaned Tokens: ['get', 'free', 'access', 'now!!']
Cleaned Sentence: get free access now!!
Total Tokens   : 4
Unique Tokens  : 4
Average Length : 4.5
--------------------------------------------------
Test 2
Original Text : I absolutely looooved this product 😍😍
Cleaned Tokens: ['absolutely', 'looved', 'this', 'product']
Cleaned Sentence: absolutely looved this product
Total Tokens   : 4
Unique Tokens  : 4
Average Length : 6.75
--------------------------------------------------
Test 3
Original Text : Worst service ever... 0/10
Cleaned Tokens: ['worst', 'service', 'ever..']
Cleaned Sentence: worst service ever..
Total Tokens   : 3
Unique Tokens  : 3
Average Length : 6.0
--------------------------------------------------
Test 4
Original Text : Call me at 9876543210
Cleaned Tokens: ['call']
Cleaned Sentence: call
Total Tokens   : 1
Unique Tokens  : 1
Average Length : 4.0
--------------------------------------------------
Test 5
Original Te

# **Task 5: Frequency Analysis**

In [4]:
from collections import Counter

all_tokens = []
for item in analytics:
    all_tokens.extend(item['tokens'])

# Count frequency of each token
token_counts = Counter(all_tokens)

# Top 10 most frequent words
top_10 = token_counts.most_common(10)

# sort by frequency ascending and take first 5
least_5 = sorted(token_counts.items(), key=lambda x: x[1])[:5]

print("Top 10 Most Frequent Words:")
for word, freq in top_10:
    print(f"{word}: {freq}")

print("\nTop 5 Least Frequent Words:")
for word, freq in least_5:
    print(f"{word}: {freq}")


Top 10 Most Frequent Words:
this: 4
now!!: 2
get: 1
free: 1
access: 1
absolutely: 1
looved: 1
product: 1
worst: 1
service: 1

Top 5 Least Frequent Words:
get: 1
free: 1
access: 1
absolutely: 1
looved: 1


# **Task 6: Build Full Pipeline**

In [5]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        # preprocess
        cleaned = preprocess_text(text)
        clean_sentences.append(cleaned)

        tokens = cleaned.split()
        all_tokens.extend(tokens)
    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

pipeline_output = full_pipeline(sample_inputs)

print("All Tokens:", pipeline_output["tokens"])
print("\nCleaned Sentences:")
for sent in pipeline_output["clean_sentences"]:
    print("-", sent)

All Tokens: ['get', 'free', 'access', 'now!!', 'absolutely', 'looved', 'this', 'product', 'worst', 'service', 'ever..', 'call', 'this', 'the', 'best', 'course!!', 'visit', 'now!', 'noo', 'this', 'baad!!', 'got', 'win', 'now!!', 'limited', 'offer!!', 'not', 'happy', 'with', 'this']

Cleaned Sentences:
- get free access now!!
- absolutely looved this product
- worst service ever..
- call
- this the best course!!
- visit now!
- noo this baad!!
- got
- win now!! limited offer!!
- not happy with this


# **Task 7: Error Handling**

In [6]:
def reduce_repeats(word: str) -> str:
    result = []
    for char in word:
        # If the last two characters are the same as current, skip
        if len(result) >= 2 and result[-1] == char and result[-2] == char:
            continue
        result.append(char)
    return "".join(result)

def preprocess_text(text: str) -> str:
    # 1. skip URLs and email-like patterns
    words = text.split()
    cleaned_words = []
    for word in words:
        if word.startswith("http") or word.startswith("www") or "@" in word:
            continue
        cleaned_words.append(word)
    text = " ".join(cleaned_words)

    # skip numbers
    text = "".join([ch for ch in text if not ch.isdigit()])

    # lowercase
    text = text.lower()

    # Handle repeated characters
    words = text.split()
    words = [reduce_repeats(word) for word in words]

    # Skip short tokens (≤ 2), except 'no' and 'not'
    meaningful = {"no", "not"}
    words = [word for word in words if len(word) > 2 or word in meaningful]

    # Remove extra spaces
    text = " ".join(words).strip()
    return text

# Error case
test_cases = [
    "", # Empty string
    "😂😂😂👏👏👏", # Only emojis
    "12345", # Only numbers
    "This is soooo goooood!!!", # Normal repeated characters
    "Visit http://sample.com now", # url
]

for i, case in enumerate(test_cases, start=1):
    cleaned = preprocess_text(case)
    print(f"Test {i}")
    print("Original:", case)
    print("Cleaned :", cleaned if cleaned else "(empty after cleaning)")
    print("-" * 40)

Test 1
Original: 
Cleaned : (empty after cleaning)
----------------------------------------
Test 2
Original: 😂😂😂👏👏👏
Cleaned : 😂😂👏👏
----------------------------------------
Test 3
Original: 12345
Cleaned : (empty after cleaning)
----------------------------------------
Test 4
Original: This is soooo goooood!!!
Cleaned : this soo good!!
----------------------------------------
Test 5
Original: Visit http://sample.com now
Cleaned : visit now
----------------------------------------
